# Fase 1 - Carga y Preprocesamiento del Dataset IroSvA
Se aplica la secuencia de preprocesamiento común a todos los modelos y se extraen
las 9 variables lingüísticas como features numéricos. Se utiliza la partición oficial
train/test del dataset (Ortega-Bueno et al., 2019).
Se generan archivos para tres variantes de texto: normal, con stemming y con lematización,
tanto combinados (3 variantes dialectales) como por variante individual.

## 1. Importación de librerías

In [1]:
import pandas as pd
import re
import emoji
import unicodedata
import nltk
from nltk.stem import SnowballStemmer
import spacy
from tqdm import tqdm
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

True

## 2. Carga y unión del conjunto de entrenamiento
Se combinan las tres variantes dialectales (México, España, Cuba) y se eliminan
los 3 duplicados identificados en la fase de exploración.

In [2]:
df_mx_train = pd.read_csv('../data/irosva.mx.training.csv')
df_es_train = pd.read_csv('../data/irosva.es.training.csv')
df_cu_train = pd.read_csv('../data/irosva.cu.training.csv')

df_mx_train['VARIANTE'] = 'mx'
df_es_train['VARIANTE'] = 'es'
df_cu_train['VARIANTE'] = 'cu'

df_train = pd.concat([df_mx_train, df_es_train, df_cu_train], ignore_index=True)
antes = len(df_train)
df_train = df_train.drop_duplicates(subset='MESSAGE').reset_index(drop=True)
eliminados = antes - len(df_train)
print(f'Train: {antes} → {len(df_train)} registros ({eliminados} duplicados eliminados)')
print(f'Registros por variante: {df_train["VARIANTE"].value_counts().to_dict()}')

Train: 7200 → 7197 registros (3 duplicados eliminados)
Registros por variante: {'cu': 2400, 'mx': 2399, 'es': 2398}


## 3. Carga y combinación del conjunto de test
Los archivos de test del dataset IroSvA separan el texto de las etiquetas reales
en dos archivos distintos. Se combinan por ID antes de unir las variantes.

In [3]:
def cargar_test(variante, nombre_variante):
    df_texto = pd.read_csv(f'../data/irosva.{variante} - irosva.{variante}.test.csv')
    df_truth = pd.read_csv(f'../data/irosva.{variante}.test.truth.csv')
    df_texto = df_texto.drop(columns=['IS_IRONIC'])
    df = df_texto.merge(df_truth[['ID', 'IS_IRONIC']], on='ID')
    df['VARIANTE'] = nombre_variante
    return df

df_test = pd.concat([
    cargar_test('mx', 'mx'),
    cargar_test('es', 'es'),
    cargar_test('cu', 'cu')
], ignore_index=True)

dups_test = df_test.duplicated(subset='MESSAGE').sum()
print(f'Test: {len(df_test)} registros | {dups_test} duplicados en MESSAGE')
print(f'Distribución IS_IRONIC: {df_test["IS_IRONIC"].value_counts().to_dict()}')
print(f'Registros por variante: {df_test["VARIANTE"].value_counts().to_dict()}')

Test: 1800 registros | 0 duplicados en MESSAGE
Distribución IS_IRONIC: {0: 1201, 1: 599}
Registros por variante: {'mx': 600, 'es': 600, 'cu': 600}


## 4. Extracción de variables lingüísticas

Se extraen **9 variables lingüísticas** del texto original **antes del preprocesamiento**.
Las features son idénticas para las tres variantes de texto (normal, stem, lemma)
porque se calculan una sola vez sobre el texto original.

(González-Ibáñez et al., 2011; Joshi et al., 2015, 2017; Šandor & Bagić Babac, 2023; RAE, 2011)

In [4]:
def quitar_tildes(texto):
    return ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )

NEGACIONES = {
    'no', 'nunca', 'jamas', 'tampoco',
    'nada', 'nadie',
    'ningun', 'ninguno', 'ninguna', 'ningunos', 'ningunas',
    'ni', 'sino', 'sin'
}

def extraer_features_linguisticos(texto):
    texto_str  = str(texto)
    texto_norm = quitar_tildes(texto_str.lower())
    palabras   = re.findall(r'\b\w+\b', texto_norm)
    n_exc = texto_str.count('!') + texto_str.count('¡')
    n_int = texto_str.count('?') + texto_str.count('¿')
    n_may = len(re.findall(r'\b[A-ZÁÉÍÓÚÜÑ]{3,}\b', texto_str))
    n_emo = sum(1 for c in texto_str if c in emoji.EMOJI_DATA)
    n_ris = len(re.findall(r'\b(ja+ja+|je+je+|ji+ji+|ha+ha+|xs+|xd+|lol+|jsjs)\b', texto_norm))
    n_neg = sum(1 for p in palabras if p in NEGACIONES)
    n_elo = len(re.findall(r'(.)\1{2,}', texto_norm))
    n_com = len(re.findall(r'["\'‘’“”«»]', texto_str))
    n_pun = texto_str.count('...') + texto_str.count('…')
    return [n_exc, n_int, n_may, n_emo, n_ris, n_neg, n_elo, n_com, n_pun]

FEATURE_COLS = ['n_exc','n_int','n_may','n_emo','n_ris','n_neg','n_elo','n_com','n_pun']

for df in [df_train, df_test]:
    feats = df['MESSAGE'].apply(extraer_features_linguisticos).tolist()
    for i, col in enumerate(FEATURE_COLS):
        df[col] = [f[i] for f in feats]

print(f'Features extraídos: {FEATURE_COLS}')
print('\n=== Medias features — Train ===')
print(df_train[FEATURE_COLS].mean().round(3))
print('\n=== Medias features — Test ===')
print(df_test[FEATURE_COLS].mean().round(3))

Features extraídos: ['n_exc', 'n_int', 'n_may', 'n_emo', 'n_ris', 'n_neg', 'n_elo', 'n_com', 'n_pun']

=== Medias features — Train ===
n_exc    0.401
n_int    0.436
n_may    0.657
n_emo    0.222
n_ris    0.006
n_neg    0.686
n_elo    0.292
n_com    0.190
n_pun    0.121
dtype: float64

=== Medias features — Test ===
n_exc    0.327
n_int    0.488
n_may    0.611
n_emo    0.178
n_ris    0.003
n_neg    0.699
n_elo    0.317
n_com    0.186
n_pun    0.141
dtype: float64


## 5. Preprocesamiento base
Genera MESSAGE_CLEAN: texto limpio sin puntuación, en minúsculas,
con emojis convertidos a texto y caracteres repetidos normalizados.
Este texto es la base para las tres variantes de vectorización.

In [5]:
def preprocesar(texto):
    texto = re.sub(r'http\S+|www\S+', '[URL]', texto)
    texto = re.sub(r'@\w+', '[USER]', texto)
    texto = emoji.demojize(texto, language='es')
    texto = re.sub(r'(.)\1{2,}', r'\1\1', texto)
    texto = re.sub(r'[^\w\s\[\]áéíóúüñÁÉÍÓÚÜÑ]', ' ', texto)
    texto = texto.lower()
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

df_train['MESSAGE_CLEAN'] = df_train['MESSAGE'].apply(preprocesar)
df_test['MESSAGE_CLEAN']  = df_test['MESSAGE'].apply(preprocesar)
print(f'Preprocesamiento base aplicado — Train: {df_train.shape} | Test: {df_test.shape}')

for variante in ['mx', 'es', 'cu']:
    ejemplo = df_train[df_train['VARIANTE'] == variante].iloc[0]
    print(f'\n[{variante.upper()}]')
    print(f'ORIGINAL: {ejemplo["MESSAGE"]}')
    print(f'LIMPIO:   {ejemplo["MESSAGE_CLEAN"]}')

Preprocesamiento base aplicado — Train: (7197, 15) | Test: (1800, 15)

[MX]
ORIGINAL: Rica económicamente, pero muy pobre en objetividad.
LIMPIO:   rica económicamente pero muy pobre en objetividad

[ES]
ORIGINAL: @ArmandoRuido007 @ANTI_MERMA50 @JoanTarda En vez de Joan Tarda van a llamarle “No han tarda” en callarle la boca 🤣🤣
LIMPIO:   [user] [user] [user] en vez de joan tarda van a llamarle no han tarda en callarle la boca cara_revolviéndose_de_la_risa cara_revolviéndose_de_la_risa

[CU]
ORIGINAL: magnifico
LIMPIO:   magnifico


## 6. Stemming
Aplica SnowballStemmer sobre MESSAGE_CLEAN para obtener MESSAGE_STEM.
(Evaluación experimental a petición del asesor)

In [6]:
stemmer = SnowballStemmer('spanish')

def aplicar_stemming(texto):
    return ' '.join(stemmer.stem(p) for p in str(texto).split())

print('Aplicando stemming...')
df_train['MESSAGE_STEM'] = df_train['MESSAGE_CLEAN'].apply(aplicar_stemming)
df_test['MESSAGE_STEM']  = df_test['MESSAGE_CLEAN'].apply(aplicar_stemming)

print('Ejemplos stem (clean → stem):')
for i in range(3):
    clean = df_train['MESSAGE_CLEAN'].iloc[i]
    stem  = df_train['MESSAGE_STEM'].iloc[i]
    print(f'  CLEAN: {clean[:70]}')
    print(f'  STEM:  {stem[:70]}')
    print()

Aplicando stemming...
Ejemplos stem (clean → stem):
  CLEAN: rica económicamente pero muy pobre en objetividad
  STEM:  ric econom per muy pobr en objet

  CLEAN: en algo tiene razón mafias hay en todo hasta en su 4t
  STEM:  en algo tien razon mafi hay en tod hast en su 4t

  CLEAN: de cuándo acá tan preocupados por la ciencia y la investigación
  STEM:  de cuand aca tan preocup por la cienci y la investig



## 7. Lematización
Aplica lematización con spaCy (es_core_news_sm) sobre MESSAGE_CLEAN
para obtener MESSAGE_LEMMA.
(Evaluación experimental a petición del asesor)

Si no tienes el modelo instalado ejecuta:
`!python -m spacy download es_core_news_sm`

In [7]:
print('Cargando modelo spaCy...')
nlp = spacy.load('es_core_news_sm', disable=['parser', 'ner'])
print('Modelo cargado.')

def lematizar_corpus(textos, batch_size=256):
    lemas = []
    for doc in tqdm(nlp.pipe(textos, batch_size=batch_size), total=len(textos), desc='Lematizando'):
        lemas.append(' '.join(t.lemma_ for t in doc if not t.is_space))
    return lemas

print('\nAplicando lematización a train...')
df_train['MESSAGE_LEMMA'] = lematizar_corpus(df_train['MESSAGE_CLEAN'].tolist())

print('Aplicando lematización a test...')
df_test['MESSAGE_LEMMA'] = lematizar_corpus(df_test['MESSAGE_CLEAN'].tolist())

print('\nEjemplos lemma (clean → lemma):')
for i in range(3):
    clean = df_train['MESSAGE_CLEAN'].iloc[i]
    lemma = df_train['MESSAGE_LEMMA'].iloc[i]
    print(f'  CLEAN: {clean[:70]}')
    print(f'  LEMMA: {lemma[:70]}')
    print()

Cargando modelo spaCy...
Modelo cargado.

Aplicando lematización a train...


Lematizando: 100%|██████████| 7197/7197 [00:11<00:00, 599.90it/s]


Aplicando lematización a test...


Lematizando: 100%|██████████| 1800/1800 [00:03<00:00, 593.29it/s]


Ejemplos lemma (clean → lemma):
  CLEAN: rica económicamente pero muy pobre en objetividad
  LEMMA: rico económicamente pero mucho pobre en objetividad

  CLEAN: en algo tiene razón mafias hay en todo hasta en su 4t
  LEMMA: en algo tener razón mafia haber en todo hasta en su 4 t

  CLEAN: de cuándo acá tan preocupados por la ciencia y la investigación
  LEMMA: de cuándo acá tanto preocupado por el ciencia y el investigación



## 7.5 Análisis de vocabulario post-preprocesamiento
Se analiza el tamaño del vocabulario efectivo después del preprocesamiento
para determinar el valor óptimo de `max_features` en TF-IDF.
Se evalúan las tres variantes de texto (normal, stem, lemma) con distintos
umbrales de `min_df` sobre el conjunto de entrenamiento.

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

variantes_texto = [
    ('normal', 'MESSAGE_CLEAN'),
    ('stem',   'MESSAGE_STEM'),
    ('lemma',  'MESSAGE_LEMMA'),
]

print(f'{"Variante texto":12} {"min_df":>8} {"Vocabulario":>13} {"Retenido %":>12}')
print('-' * 50)

vocab_sizes = {}
for nombre, col in variantes_texto:
    corpus_train = df_train[col].tolist()
    # sin filtro — vocabulario bruto post-preprocesamiento
    vect_raw = TfidfVectorizer(min_df=1, max_df=0.95)
    vect_raw.fit(corpus_train)
    vocab_raw = len(vect_raw.vocabulary_)
    vocab_sizes[nombre] = {}

    for min_df in [1, 2, 3, 5]:
        vect = TfidfVectorizer(min_df=min_df, max_df=0.95)
        vect.fit(corpus_train)
        v = len(vect.vocabulary_)
        vocab_sizes[nombre][min_df] = v
        pct = v / vocab_raw * 100
        print(f'{nombre:12} {min_df:>8} {v:>13,} {pct:>11.1f}%')
    print()

Variante texto   min_df   Vocabulario   Retenido %
--------------------------------------------------
normal              1        19,293       100.0%
normal              2         8,027        41.6%
normal              3         5,297        27.5%
normal              5         3,233        16.8%

stem                1        10,803       100.0%
stem                2         5,252        48.6%
stem                3         3,835        35.5%
stem                5         2,631        24.4%

lemma               1        14,779       100.0%
lemma               2         6,136        41.5%
lemma               3         4,168        28.2%
lemma               5         2,634        17.8%



In [9]:
# Decisión de max_features basada en el vocabulario con min_df=2
print('Vocabulario efectivo con min_df=2 por variante de texto:')
print('(Este es el tamaño real de vocabulario útil para TF-IDF)\n')

for nombre, col in variantes_texto:
    corpus_train = df_train[col].tolist()
    vect = TfidfVectorizer(min_df=2, max_df=0.95)
    vect.fit(corpus_train)
    v = len(vect.vocabulary_)
    print(f'  {nombre:8} → {v:,} términos con min_df=2')
    if v <= 10_000:
        print(f'           → max_features=None  (vocabulario cabe completo)')
    elif v <= 15_000:
        print(f'           → max_features={v}  (usar vocabulario completo)')
    else:
        print(f'           → max_features=15000  (recortar por memoria/velocidad)')
    print()

Vocabulario efectivo con min_df=2 por variante de texto:
(Este es el tamaño real de vocabulario útil para TF-IDF)

  normal   → 8,027 términos con min_df=2
           → max_features=None  (vocabulario cabe completo)

  stem     → 5,252 términos con min_df=2
           → max_features=None  (vocabulario cabe completo)

  lemma    → 6,136 términos con min_df=2
           → max_features=None  (vocabulario cabe completo)



## 8. Exportación de todos los datasets

Se exportan 24 archivos: 3 tipos de texto × 4 datasets × 2 splits (train/test).
Todos comparten las mismas 9 features lingüísticas, solo varía MESSAGE_CLEAN.

En 02_ml_modelos.ipynb se elige el tipo de texto con el parámetro `PREP`.

In [10]:
def exportar_variante(df_tr, df_te, col_texto, sufijo):
    """
    Exporta train y test para un tipo de preprocesamiento.
    Copia el texto indicado en MESSAGE_CLEAN para mantener
    compatibilidad con 02_ml_modelos.ipynb.
    """
    cols_base = ['ID','TOPIC','IS_IRONIC','MESSAGE','VARIANTE'] + FEATURE_COLS

    for split, df in [('train', df_tr), ('test', df_te)]:
        df_out = df[cols_base].copy()
        df_out['MESSAGE_CLEAN'] = df[col_texto]

        # Combinado
        df_out.to_csv(f'../data/{split}_clean{sufijo}.csv', index=False)

        # Por variante
        for cod in ['mx', 'es', 'cu']:
            df_out[df_out['VARIANTE'] == cod]\
                .reset_index(drop=True)\
                .to_csv(f'../data/{split}_clean{sufijo}_{cod}.csv', index=False)

    tipo = sufijo if sufijo else '(normal)'
    print(f'  {tipo:8} → train_clean{sufijo}.csv | train_clean{sufijo}_mx/es/cu.csv')
    print(f'  {"":8}   test_clean{sufijo}.csv  | test_clean{sufijo}_mx/es/cu.csv')

print('Exportando datasets...\n')
exportar_variante(df_train, df_test, 'MESSAGE_CLEAN', '')
exportar_variante(df_train, df_test, 'MESSAGE_STEM',  '_stem')
exportar_variante(df_train, df_test, 'MESSAGE_LEMMA', '_lemma')
print('\n✓ 24 archivos generados en ../data/')

Exportando datasets...

  (normal) → train_clean.csv | train_clean_mx/es/cu.csv
             test_clean.csv  | test_clean_mx/es/cu.csv
  _stem    → train_clean_stem.csv | train_clean_stem_mx/es/cu.csv
             test_clean_stem.csv  | test_clean_stem_mx/es/cu.csv
  _lemma   → train_clean_lemma.csv | train_clean_lemma_mx/es/cu.csv
             test_clean_lemma.csv  | test_clean_lemma_mx/es/cu.csv

✓ 24 archivos generados en ../data/


## 9. Resumen de archivos generados
```
data/
├── train_clean.csv          test_clean.csv          ← normal
├── train_clean_mx/es/cu     test_clean_mx/es/cu
├── train_clean_stem.csv     test_clean_stem.csv     ← stemming
├── train_clean_stem_mx/es/cu  test_clean_stem_mx/es/cu
├── train_clean_lemma.csv    test_clean_lemma.csv    ← lematización
└── train_clean_lemma_mx/es/cu test_clean_lemma_mx/es/cu
```
En 02_ml_modelos.ipynb cambiar: `PREP = 'normal'` | `'stem'` | `'lemma'`